# Blok 12: Wstęp do AI (Machine Learning)

W tym notatniku znajdziesz zestaw zadań do bloku:
- od zadań koncepcyjnych (jak zaklasyfikować problem),
- przez inżynierię cech i metryki,
- aż po pierwszy model w bibliotece `scikit-learn`.

Rozwiązania są przygotowane w osobnym notatniku `cwiczenia_solved.ipynb`.

Zawsze czytaj dokładnie treść zadania (komórka markdown nad kodem) i uzupełniaj miejsca oznaczone komentarzami `# TODO`.


## Opis datasetu

W tym bloku korzystamy z **syntetycznych** (wymyślonych) mini-zbiorów danych, zdefiniowanych bezpośrednio w Pythonie.

### 1. Dataset: `customers_data` – klienci sklepu online (klasyfikacja)

Każdy wiersz reprezentuje jednego użytkownika wizytującego sklep.

Kolumny (klucze w słownikach):
- `age` (int) – wiek użytkownika,
- `country` (str) – kraj pochodzenia (np. "PL", "DE", "UK"),
- `device` (str) – typ urządzenia (np. "mobile", "desktop"),
- `source` (str) – źródło ruchu (np. "ads", "organic", "email"),
- `purchase` (int, 0/1) – czy użytkownik dokonał zakupu (1) czy nie (0).

Zadanie:
- przewidzieć `purchase` na podstawie cech klienta – klasyfikacja binarna.

### 2. Dataset: `apartments_data` – mieszkania (regresja)

Każdy wiersz reprezentuje mieszkanie na sprzedaż.

Kolumny:
- `rooms` (int) – liczba pokoi,
- `area` (float) – powierzchnia mieszkania w metrach kwadratowych,
- `district` (str) – dzielnica (np. "center", "suburbs"),
- `price` (float) – cena mieszkania w tysiącach (np. 350 oznacza 350 000).

Zadanie:
- przewidzieć `price` na podstawie `rooms`, `area`, `district` – regresja.

### 3. Dataset: `churn_true` i `churn_pred` – wyniki modelu churn (klasyfikacja)

Mały, sztuczny przykład do nauki confusion matrix i metryk klasyfikacji.

Listy:
- `churn_true` – prawdziwe etykiety (czy klient odszedł: 1 – odszedł, 0 – został),
- `churn_pred` – przewidywania modelu (taki sam format 0/1).

Na tych listach będziesz liczyć TN, FP, FN, TP i metryki: accuracy, precision, recall, F1.

Dodatkowo w niektórych zadaniach możesz tworzyć swoje własne mini-zbiory, aby lepiej zrozumieć pojęcia.

W kolejnej komórce znajduje się kod, który definiuje te przykładowe dane. Uruchom ją na początku pracy z notebookiem.


In [14]:
# Przygotowanie przykładowych danych syntetycznych
# Uruchom tę komórkę na początku (Shift+Enter)

import numpy as np
import pandas as pd

# Dataset 1: klienci sklepu online
customers_data = [
    {"age": 22, "country": "PL", "device": "mobile",  "source": "ads",     "purchase": 1},
    {"age": 35, "country": "PL", "device": "desktop", "source": "organic", "purchase": 0},
    {"age": 28, "country": "DE", "device": "mobile",  "source": "email",   "purchase": 1},
    {"age": 42, "country": "UK", "device": "desktop", "source": "ads",     "purchase": 0},
    {"age": 30, "country": "PL", "device": "mobile",  "source": "organic", "purchase": 1},
    {"age": 50, "country": "DE", "device": "desktop", "source": "email",   "purchase": 0},
    {"age": 27, "country": "PL", "device": "mobile",  "source": "ads",     "purchase": 1},
    {"age": 45, "country": "UK", "device": "desktop", "source": "organic", "purchase": 0}
]

customers_df = pd.DataFrame(customers_data)

# Dataset 2: mieszkania
apartments_data = [
    {"rooms": 2, "area": 38.0, "district": "center",  "price": 360.0},
    {"rooms": 3, "area": 54.0, "district": "center",  "price": 520.0},
    {"rooms": 2, "area": 45.0, "district": "suburbs", "price": 310.0},
    {"rooms": 4, "area": 70.0, "district": "suburbs", "price": 480.0},
    {"rooms": 1, "area": 28.0, "district": "center",  "price": 290.0},
    {"rooms": 3, "area": 60.0, "district": "suburbs", "price": 430.0}
]

apartments_df = pd.DataFrame(apartments_data)

# Dataset 3: wyniki modelu churn (prawda vs predykcja)
churn_true = [0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0]
churn_pred = [0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1]

print("customers_df (pierwsze wiersze):")
display(customers_df.head())

print("apartments_df (pierwsze wiersze):")
display(apartments_df.head())

print("churn_true:", churn_true)
print("churn_pred:", churn_pred)


customers_df (pierwsze wiersze):


,age,country,device,source,purchase
0,22,PL,mobile,ads,1
1,35,PL,desktop,organic,0
2,28,DE,mobile,email,1
3,42,UK,desktop,ads,0
4,30,PL,mobile,organic,1


apartments_df (pierwsze wiersze):


,rooms,area,district,price
0,2,38.0,center,360.0
1,3,54.0,center,520.0
2,2,45.0,suburbs,310.0
3,4,70.0,suburbs,480.0
4,1,28.0,center,290.0


churn_true: [0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0]
churn_pred: [0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1]


# Część A – Typy problemów ML i podstawowe pojęcia

W zadaniach 1–4 skupisz się na zrozumieniu:
- czy dany problem to klasyfikacja, regresja czy coś innego,
- co jest cechą (`X`), a co etykietą (`y`),
- czym różni się uczenie nadzorowane od nienadzorowanego,
- jak wygląda prosty workflow projektu ML.


## Zadanie 1 – Klasyfikacja czy regresja?

Opis biznesowy:
- Masz kilka opisów problemów z różnych firm (e-commerce, finanse, marketing itd.).
- Dla każdego z nich zdecyduj, czy to **klasyfikacja**, **regresja**, czy **inne** (np. klasteryzacja, szeregi czasowe).

Cel techniczny:
- Utrwalenie rozróżnienia między klasyfikacją i regresją oraz rozpoznawanie innych typów zadań.

Poziom: 1 (łatwe)

Szacowany czas: ok. 10 minut

Wskazówki:
- Zwróć uwagę, czy wynik jest **liczbą ciągłą** czy **kategorią**.
- Jeśli wynik jest sekwencją w czasie (np. prognoza na kolejne dni), możesz potraktować to jako szeregi czasowe.
- Jeśli celem jest podział na grupy bez znanych etykiet, to klasteryzacja.

Instrukcja:
- Uzupełnij słownik `problems` typem zadania w polu `task_type`.
- Możesz zapisać odpowiedzi jako tekst (np. "classification", "regression", "clustering", "time_series").


In [15]:
# Zadanie 1 – Klasyfikacja czy regresja?

# Poniżej kilka opisów problemów. Uzupełnij typ zadania w polu "task_type".
# Użyj wartości np. "classification", "regression", "clustering", "time_series".

problems = [
    {
        "name": "Churn prediction",
        "description": "Przewidywanie, czy klient zrezygnuje z usługi w ciągu najbliższego miesiąca.",
        "task_type": "classification"
    },
    {
        "name": "House price prediction",
        "description": "Przewidywanie ceny mieszkania na podstawie liczby pokoi, metrażu i lokalizacji.",
        "task_type": "regression"
    },
    {
        "name": "Customer segmentation",
        "description": "Podział klientów na grupy o podobnym zachowaniu zakupowym bez znanych etykiet.",
        "task_type": "clustering"
    },
    {
        "name": "Sales forecast",
        "description": "Prognozowanie dziennej sprzedaży sklepu na najbliższe 30 dni.",
        "task_type": "time_series"
    }
]

for p in problems:
    print(p["name"], "->", p["task_type"])


Churn prediction -> classification
House price prediction -> regression
Customer segmentation -> clustering
Sales forecast -> time_series


## Zadanie 2 – Cechy (X) vs etykieta (y)

Opis biznesowy:
- Pracujesz nad modelem dla sklepu internetowego.
- Masz mini-dataset `customers_df` z cechami użytkowników i informacją, czy dokonali zakupu.

Cel techniczny:
- Zidentyfikować, które kolumny są cechami (`X`), a która kolumna jest etykietą (`y`).
- Przygotować obiekty `X` i `y` w postaci odpowiednich struktur (np. `DataFrame` i `Series`).

Poziom: 1 (łatwe)

Szacowany czas: ok. 10 minut

Wskazówki:
- Zastanów się, co chcesz przewidywać na podstawie jakich informacji.
- Zapisz pełne zdanie: "Chcemy przewidzieć ... na podstawie ..." i dopiero potem zrób wybór.

Instrukcja:
- Wybierz kolumnę docelową (etykietę) i kolumny cech.
- Utwórz `X_customers` i `y_customers` na bazie `customers_df`.


In [16]:
customers_df

,age,country,device,source,purchase
0,22,PL,mobile,ads,1
1,35,PL,desktop,organic,0
2,28,DE,mobile,email,1
3,42,UK,desktop,ads,0
4,30,PL,mobile,organic,1
5,50,DE,desktop,email,0
6,27,PL,mobile,ads,1
7,45,UK,desktop,organic,0


In [17]:
# Zadanie 2 – Cechy vs etykieta

# np. "Chcemy przewidzieć wiek klienta na podstawie kupna"

X_customers = customers_df[["age", "country", "device", "source"]]
y_customers = customers_df["purchase"]

print("X_customers shape:", None if X_customers is None else X_customers.shape)
print("y_customers shape:", None if y_customers is None else y_customers.shape)


X_customers shape: (8, 4)
y_customers shape: (8,)


## Zadanie 3 – Supervised vs unsupervised

Opis biznesowy:
- Masz zestaw scenariuszy analitycznych.
- Dla każdego określ, czy to przykład **uczenia nadzorowanego (supervised)**, czy **nienadzorowanego (unsupervised)**.

Cel techniczny:
- Zrozumieć, że supervised oznacza dostęp do etykiet (`y`), a unsupervised – ich brak.

Poziom: 1 (łatwe)

Szacowany czas: ok. 10 minut

Instrukcja:
- Uzupełnij pole `learning_type` dla każdego scenariusza.
- Użyj wartości np. "supervised" lub "unsupervised".
- Zwróć uwagę, czy w opisie jest informacja o historycznych odpowiedziach.


In [18]:
# Zadanie 3 – Supervised vs unsupervised

scenarios = [
    {
        "description": "Mamy dane historyczne: cechy klienta i informację, czy zrezygnował z usługi. Chcemy przewidywać churn.",
        "learning_type": "supervised"
    },
    {
        "description": "Mamy tylko dane o zachowaniu klientów w sklepie (odsłony, kliknięcia, ścieżki). Chcemy znaleźć segmenty klientów.",
        "learning_type": "unsupervised"
    },
    {
        "description": "Mamy dane o transakcjach i informację, czy były fraudem. Chcemy nauczyć model wykrywania fraudu.",
        "learning_type": "supervised"
    }
]

for s in scenarios:
    print("Opis:", s["description"])
    print("Rodzaj uczenia:", s["learning_type"])
    print("-")

Opis: Mamy dane historyczne: cechy klienta i informację, czy zrezygnował z usługi. Chcemy przewidywać churn.
Rodzaj uczenia: supervised
-
Opis: Mamy tylko dane o zachowaniu klientów w sklepie (odsłony, kliknięcia, ścieżki). Chcemy znaleźć segmenty klientów.
Rodzaj uczenia: unsupervised
-
Opis: Mamy dane o transakcjach i informację, czy były fraudem. Chcemy nauczyć model wykrywania fraudu.
Rodzaj uczenia: supervised
-


## Zadanie 4 – Workflow ML na prostym case

Opis biznesowy:
- Sklep online chce przewidywać, czy użytkownik kupi produkt podczas wizyty.
- Chcesz wypisać kroki workflow ML dla tego problemu.

        
Cel techniczny:
- Utrwalić etapy pracy z modelem: od problemu biznesowego po ocenę modelu.

Poziom: 1–2 (łatwe/średnie)

Szacowany czas: ok. 10 minut

Instrukcja:
- Wypełnij listę `workflow_steps` krokami, jakie wykonałbyś w takim projekcie.
- Możesz opisać kroki w języku naturalnym (po polsku) jako stringi.
- Zadbaj o kolejność: problem, dane, EDA, cechy, model, metryki, trening, test, wnioski.


In [19]:
# Zadanie 4 – Workflow ML "Zdefiniowanie problemu biznesowego: przewidzieć, czy użytkownik kupi produkt."

workflow_steps = [
    "Zdefiniować problem - jaki? cel biznesowy?",
    "Zadbać o dane / dobranie metryk",
    "EDA - statystyka / rozkłady / klas statystyka",
    "Preprocessing",
    "Cechy wyjściowe",
    "Przygotowanie cech, OHE / skalowanie cech",
    "Wybór modelu / metryk",
    "Trening modelu",
    "Ocena modelu",
    "Kolejna iteracja - workflow"
]

for i, step in enumerate(workflow_steps, start=1):
    print(f"Krok {i}:", step)


Krok 1: Zdefiniować problem - jaki? cel biznesowy?
Krok 2: Zadbać o dane / dobranie metryk
Krok 3: EDA - statystyka / rozkłady / klas statystyka
Krok 4: Preprocessing
Krok 5: Cechy wyjściowe
Krok 6: Przygotowanie cech, OHE / skalowanie cech
Krok 7: Wybór modelu / metryk
Krok 8: Trening modelu
Krok 9: Ocena modelu
Krok 10: Kolejna iteracja - workflow


# Część B – Inżynieria cech

W zadaniach 5–6 pracujesz na mini-datasetach, aby zrozumieć:
- jak działa one-hot encoding dla zmiennych kategorycznych,
- jak działa skalowanie cech: min-max i standaryzacja.

Skup się na **intuicji**, nie na zapamiętywaniu wzorów na pamięć.


## Zadanie 5 – One-hot encoding na małym zbiorze

Opis biznesowy:
- Zmienna `device` w `customers_df` przyjmuje wartości "mobile" lub "desktop".
- Model potrzebuje wartości numerycznych, więc musimy zakodować tę zmienną.

Cel techniczny:
- Zrozumieć ideę one-hot encodingu.
- Zastosować one-hot encoding w Pythonie (np. `pandas.get_dummies`).

Poziom: 2 (średnie)

Szacowany czas: ok. 15 minut

Instrukcja:
- Najpierw spróbuj ręcznie rozpisać, jakie kolumny powstaną po one-hot encodingu dla `device`.
- Następnie użyj `pandas` do stworzenia zakodowanej wersji dataframe.
- Możesz dodać także one-hot encoding dla `country` lub `source` (opcjonalnie).

Podpowiedź:
- `pd.get_dummies(customers_df, columns=["device"])` może być dobrym startem.


In [20]:
# Zadanie 5 – One-hot encoding

# Kolumny do one-hot encodingu: device_mobile, device_desktop, country, source.

display(customers_df)
customers_encoded = pd.get_dummies(customers_df, columns=["device", "country", "source"], drop_first=False)

print("Oryginalne kolumny:", list(customers_df.columns))
print("Zakodowane kolumny:", None if customers_encoded is None else list(customers_encoded.columns))

,age,country,device,source,purchase
0,22,PL,mobile,ads,1
1,35,PL,desktop,organic,0
2,28,DE,mobile,email,1
3,42,UK,desktop,ads,0
4,30,PL,mobile,organic,1
5,50,DE,desktop,email,0
6,27,PL,mobile,ads,1
7,45,UK,desktop,organic,0


Oryginalne kolumny: ['age', 'country', 'device', 'source', 'purchase']
Zakodowane kolumny: ['age', 'purchase', 'device_desktop', 'device_mobile', 'country_DE', 'country_PL', 'country_UK', 'source_ads', 'source_email', 'source_organic']


## Zadanie 6 – Skalowanie i standaryzacja cech

Opis biznesowy:
- W datasetcie `apartments_df` cechy `area` (metraż) i `rooms` (liczba pokoi) mają różne skale.
- Chcesz porównać efekt min-max scalingu i standaryzacji.

Cel techniczny:
- Przećwiczyć obliczanie min-max scalingu i standaryzacji na małym zbiorze.

Poziom: 2 (średnie)

Szacowany czas: ok. 20 minut

Instrukcja:
- Wyciągnij kolumnę `area` i oblicz dla niej:
  - min,
  - max,
  - średnią,
  - odchylenie standardowe.
- Zastosuj min-max scaling i standaryzację do wszystkich wierszy tej kolumny.
- Powtórz analogicznie dla `rooms` (opcjonalnie lub skróconą wersję).

Podpowiedź (wzory):
- min-max: (x - x_min) / (x_max - x_min)
- standaryzacja: (x - mean) / std

Możesz użyć `numpy` lub pisać wzory "ręcznie" w Pythonie.


In [21]:
# Zadanie 6 – Skalowanie i standaryzacja

area = apartments_df["area"]
rooms = apartments_df["rooms"]

area_min = area.min()
area_max = area.max()
area_mean = area.mean()
area_std = area.std()

print("area_min:", area_min)
print("area_max:", area_max)
print("area_mean:", area_mean)
print("area_std:", area_std)

area_minmax = (area - area_min) / (area_max - area_min)
rooms_minmax = (rooms - rooms.min()) / (rooms.max() - rooms.min())
area_standard = (area - area_mean) / area_std
rooms_standard = (rooms - rooms.mean()) / rooms.std()


print("\nOryginalne area:")
print(area.values)
print("\nOryginalne rooms:")
print(rooms.values)

print("\narea po min-max scalingu:")
print(area_minmax)
print("\nrooms po min-max scalingu:")
print(rooms_minmax)

print("\narea po standaryzacji:")
print(area_standard)
print("\nrooms po standaryzacji:")
print(rooms_standard)

area_min: 28.0
area_max: 70.0
area_mean: 49.166666666666664
area_std: 15.263245613783022

Oryginalne area:
[38. 54. 45. 70. 28. 60.]

Oryginalne rooms:
[2 3 2 4 1 3]

area po min-max scalingu:
0    0.238095
1    0.619048
2    0.404762
3    1.000000
4    0.000000
5    0.761905
Name: area, dtype: float64

rooms po min-max scalingu:
0    0.333333
1    0.666667
2    0.333333
3    1.000000
4    0.000000
5    0.666667
Name: rooms, dtype: float64

area po standaryzacji:
0   -0.731605
1    0.316665
2   -0.272987
3    1.364935
4   -1.386774
5    0.709766
Name: area, dtype: float64

rooms po standaryzacji:
0   -0.476731
1    0.476731
2   -0.476731
3    1.430194
4   -1.430194
5    0.476731
Name: rooms, dtype: float64


# Część C – Metryki klasyfikacji i regresji

W zadaniach 7–9 nauczysz się obliczać i interpretować:
- confusion matrix,
- accuracy, precision, recall, F1-score,
- metryki regresji: MSE, RMSE, MAE.

Zaczniesz od liczenia na małych, ręcznych przykładach.


## Zadanie 7 – Confusion matrix, accuracy, precision, recall, F1 (ręcznie)

Opis biznesowy:
- Masz wyniki modelu, który przewiduje, czy klient zrezygnuje z usługi (`churn`).
- Prawdziwe etykiety i predykcje są zapisane w listach `churn_true` i `churn_pred`.

Cel techniczny:
- Policz ręcznie wartości TN, FP, FN, TP.
- Na ich podstawie policz accuracy, precision, recall, F1-score.

Poziom: 2 (średnie)

Szacowany czas: ok. 20 minut

Instrukcja:
- Załóż, że klasa pozytywna to `1` (klient odchodzi).
- Przejdź po elementach `churn_true` i `churn_pred` i zlicz TN, FP, FN, TP.
- Następnie oblicz metryki.

Podpowiedź:
- accuracy = (TP + TN) / (TP + TN + FP + FN)
- precision = TP / (TP + FP)
- recall = TP / (TP + FN)
- F1 = 2 * (precision * recall) / (precision + recall)


In [22]:
# Zadanie 7 – Ręczne liczenie metryk klasyfikacji

print("churn_true:", churn_true)
print("churn_pred:", churn_pred)

TN = 0
FP = 0
FN = 0
TP = 0

for yt, yp in zip(churn_true, churn_pred):
    if yt == 1 and yp == 1:
        TP += 1
    elif yt == 0 and yp == 1:
        FP += 1
    elif yt == 1 and yp == 0:
        FN += 1
    else:
        TN += 1
    
print("TN:", TN, "FP:", FP, "FN:", FN, "TP:", TP)

total = TP + TN + FP + FN

accuracy = (TP + TN) / total
precision = TP / (TP + FP)
recall = TP / (TP + FN)
f1 = 2 * (precision * recall) / (precision + recall)

print("Accuracy:", accuracy)
print("Precision:", precision)
print("Recall:", recall)
print("F1-score:", f1)


churn_true: [0, 0, 1, 1, 0, 1, 0, 1, 1, 0, 1, 0]
churn_pred: [0, 1, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1]
TN: 4 FP: 2 FN: 2 TP: 4
Accuracy: 0.6666666666666666
Precision: 0.6666666666666666
Recall: 0.6666666666666666
F1-score: 0.6666666666666666


## Zadanie 8 – Interpretacja confusion matrix (model fraud)

Opis biznesowy:
- Masz dwa modele wykrywania fraudu w płatnościach online.
- Każdy ma inną confusion matrix.
- Musisz zdecydować, który model jest lepszy z perspektywy biznesowej.

Cel techniczny:
- Zrozumieć kompromis między precision i recall.

Poziom: 2 (średnie)

Szacowany czas: ok. 15 minut

Instrukcja:
- Poniżej zadeklarowane są dwie macierze dla modeli A i B.
- Policz precision i recall dla obu modeli (dla klasy pozytywnej = fraud).
- Zastanów się, który model jest korzystniejszy w kontekście fraudów i dlaczego.
- Swoją odpowiedź uzasadnij w komentarzu.


In [23]:
# Zadanie 8 – Interpretacja confusion matrix

# Macierz pomyłek dla modelu A (kolejność: TN, FP, FN, TP)
TN_A, FP_A, FN_A, TP_A = 980, 10, 15, 5

# Macierz pomyłek dla modelu B
TN_B, FP_B, FN_B, TP_B = 950, 40, 5, 25

precision_A = TP_A / (TP_A + FP_A) if (TP_A + FP_A) > 0 else 0
recall_A = TP_A / (TP_A + FN_A) if (TP_A + FN_A) > 0 else 0

precision_B = TP_B / (TP_B + FP_B) if (TP_B + FP_B) > 0 else 0
recall_B = TP_B / (TP_B + FN_B) if (TP_B + FN_B) > 0 else 0

print("Model A – precision:", round(precision_A, 2), "recall:", round(recall_A, 2))
print("Model B – precision:", round(precision_B, 2), "recall:", round(recall_B, 2))

# NAPISAĆ w komentarzu opisz, który model wolisz i dlaczego z perspektywy wykrywania fraudu.
# Zwróć uwagę, co jest bardziej kosztowne: fałszywy alarm czy przeoczenie fraudu.
"""Przeoczenie niesie poważną stratę finansową, 
więc interuje nas recall, wykrywa 83% oszustw. 
Precyzja wskazuje fałszywe alarmy i wynosi 62%, 
ale to jest akceptowalne (wysłanie fałszywego powiadomienia)"""


Model A – precision: 0.33 recall: 0.25
Model B – precision: 0.38 recall: 0.83


'Przeoczenie niesie poważną stratę finansową, \nwięc interuje nas recall, wykrywa 83% oszustw. \nPrecyzja wskazuje fałszywe alarmy i wynosi 62%, \nale to jest akceptowalne (wysłanie fałszywego powiadomienia)'

## Zadanie 9 – Metryki regresji: MSE, RMSE, MAE

Opis biznesowy:
- Masz prosty model, który prognozuje cenę mieszkania.
- Na kilku przykładach znasz prawdziwe ceny i przewidywania.

Cel techniczny:
- Obliczyć MSE, RMSE i MAE.
- Zrozumieć, jak te metryki reagują na duże błędy.

Poziom: 2 (średnie)

Szacowany czas: ok. 20 minut

Instrukcja:
- Załóż prosty zestaw prawdziwych cen i odpowiadających im przewidywań.
- Oblicz MSE, RMSE, MAE "ręcznie" przy użyciu Pythona lub `numpy`.
- Porównaj wyniki i zastanów się, która metryka najmocniej karze duże błędy.

Podpowiedź (wzory):
- MSE = średnia z (y - y_hat)^2
- RMSE = pierwiastek z MSE
- MAE = średnia z |y - y_hat|


In [24]:
# Zadanie 9 – Metryki regresji

y_true_prices = [360.0, 520.0, 310.0, 480.0, 290.0]
y_pred_prices = [350.0, 510.0, 400.0, 450.0, 280.0]

y_true_arr = np.array(y_true_prices)
y_pred_arr = np.array(y_pred_prices)

errors = y_true_arr - y_pred_arr
sq_errors = errors ** 2

mse = np.mean(sq_errors)
rmse = np.sqrt(mse)
mae = np.mean(np.abs(errors))

print("y_true:", y_true_prices)
print("y_pred:", y_pred_prices)
print("MSE:", mse)
print("RMSE:", rmse)
print("MAE:", mae)

"""Błąd jest zazwyczaj względem odchylenia danych, 
tutaj niewielki + to są ceny które się wachają"""

y_true: [360.0, 520.0, 310.0, 480.0, 290.0]
y_pred: [350.0, 510.0, 400.0, 450.0, 280.0]
MSE: 1860.0
RMSE: 43.12771730569565
MAE: 30.0


'Błąd jest zazwyczaj względem odchylenia danych, \ntutaj niewielki + to są ceny które się wachają'

# Część D – Pierwszy model w scikit-learn i workflow

W zadaniach 10–11 zbudujesz pierwszy prosty model w `scikit-learn` i przejdziesz przez uproszczony workflow ML.

Skup się na zrozumieniu schematu:
1. Przygotowanie X i y.
2. Podział na train/test.
3. Wybór prostego modelu.
4. `fit`, `predict`.
5. Wyliczenie podstawowej metryki.


## Zadanie 10 – Pierwszy model w scikit-learn (klasyfikacja lub regresja)

Opis biznesowy:
- Chcesz zbudować prosty model:
  - klasyfikacyjny na `customers_df` (np. przewidywanie zakupu),
  - lub regresyjny na `apartments_df` (np. przewidywanie ceny).
- Wybierz jedną z opcji (jeśli masz czas, możesz zrobić obie).

Cel techniczny:
- Przećwiczyć pełen mini-proces w `scikit-learn`:
  - `train_test_split`,
  - `fit`,
  - `predict`,
  - wyliczenie podstawowej metryki.

Poziom: 2–3 (średnie / trochę trudniejsze)

Szacowany czas: ok. 30 minut

Instrukcja (wariant klasyfikacji):
- Użyj `customers_df`.
- Przygotuj cechy (po one-hot encodingu) i etykietę `purchase`.
- Użyj `train_test_split`.
- Wybierz prosty model, np. `LogisticRegression` lub `KNeighborsClassifier`.
- Wytrenuj model i policz accuracy.

Instrukcja (wariant regresji):
- Użyj `apartments_df`.
- Przygotuj cechy `rooms`, `area` (opcjonalnie zakoduj `district`).
- Użyj `train_test_split`.
- Wybierz model, np. `LinearRegression`.
- Policz MSE / RMSE.

Wybierz jedną ścieżkę albo spróbuj obu, jeśli zdążysz.


In [33]:
# Zadanie 10 – Pierwszy model w scikit-learn

# - klasyfikacja na customers_df
# - regresja na apartments_df

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, mean_squared_error

customers_for_model = pd.get_dummies(customers_df, columns=["country", "device", "source"], drop_first=False)

X = customers_for_model.drop(columns=["purchase"])
y = customers_for_model["purchase"]

model = LogisticRegression()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=12)

model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred_train = model.predict(X_train)
acc_test = accuracy_score(y_test, y_pred)
acc_train = accuracy_score(y_train, y_pred_train)

print("Wartość metryki ACC_test:", acc_test)
print("Wartość metryki ACC_train:", acc_train)

Wartość metryki ACC_test: 1.0
Wartość metryki ACC_train: 1.0


## Zadanie 11 – Workflow ML na mini-case

Opis biznesowy:
- Chcesz podsumować cały workflow ML na przykładzie wybranego zadania (np. przewidywanie zakupu lub ceny mieszkania).

Cel techniczny:
- Połączyć wiedzę z całego bloku w spójny opis.

Poziom: 2 (średnie)

Szacowany czas: ok. 20 minut

Instrukcja:
- Wybierz jedno z zadań:
  - klasyfikacja (customers),
  - lub regresja (apartments).
- W komentarzu (lub w formie listy stringów) opisz, jak wyglądał:
  - problem biznesowy,
  - przygotowanie danych,
  - wybór cech i modelu,
  - wybór metryk,
  - trening i test,
  - interpretacja wyników.

Nie chodzi o kod, tylko o **opis słowny** workflow – możesz użyć języka naturalnego.


In [ ]:
# Zadanie 11 – Workflow ML na mini-case

# TODO: wybierz case (np. "classification_customers" albo "regression_apartments").

chosen_case = "classification_customers"  # TODO: zmień, jeśli chcesz

# TODO: opisz workflow w formie listy kroków (stringi) lub jednego dłuższego tekstu.

workflow_description = [
    "Zdefiniowanie problemu",
    "Dostep danych",
    "EDA",
    "Przygotowanie cech",
    "Podzial danych",
    "wybor modelu",
    "wybor metryk",
    "trening modelu",
    "predykcja",
    "metryk interpretacja",
    "Kolejne iteracje
]

for i, step in enumerate(workflow_description, start=1):
    print(f"Krok {i}:", step)

# Możesz też użyć komórek markdown obok, jeśli wolisz opis bardziej narracyjny.


# Część E – Praca domowa (opcjonalna) – Zadania 12 i 13

Poniższe zadania są przewidziane jako praca domowa / dla chętnych.

- Zadanie 12 – mini-pipeline klasyfikacji,
- Zadanie 13 – bias vs variance na wykresie (dla chętnych na dodatkową pracę).

Możesz je zrealizować po zajęciach, korzystając z materiałów i ściąg z tego modułu.


## Zadanie 12 – Mini-pipeline klasyfikacji (praca domowa)

Opis biznesowy:
- Masz mini-zbiór klientów sklepu (możesz użyć `customers_df` lub stworzyć własny, trochę większy).
- Chcesz zbudować prosty pipeline klasyfikacyjny przewidujący `purchase`.

Cel techniczny:
- Utrwalić pełny workflow w `scikit-learn`:
  - przygotowanie cech,
  - one-hot encoding,
  - podział train/test,
  - trening modelu,
  - wyliczenie metryk (accuracy, precision, recall, F1).

Poziom: 2–3 (średnie / trochę trudniejsze)

Szacowany czas: 30–45 minut

Instrukcja:
- Przygotuj `X` i `y` (customers + purchase).
- Wykonaj one-hot encoding zmiennych kategorycznych.
- Podziel dane na train/test.
- Wybierz model klasyfikacyjny (np. `LogisticRegression` lub `RandomForestClassifier`).
- Policz accuracy, precision, recall, F1.
- W komentarzu opisz, które metryki są dla Ciebie najważniejsze w tym case.


In [ ]:
# Zadanie 12 – Mini-pipeline klasyfikacji (praca domowa)

# TODO: zaimportuj potrzebne narzędzia (train_test_split, model, metryki)

# from sklearn.model_selection import train_test_split
# from sklearn.linear_model import LogisticRegression
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# TODO: przygotuj X i y na bazie customers_df i wykonaj one-hot encoding zmiennych kategorycznych.

X_clf = None  # TODO
y_clf = None  # TODO

# TODO: podziel na train/test
X_train_clf, X_test_clf, y_train_clf, y_test_clf = None, None, None, None

# TODO: wybierz model, wytrenuj go i policz metryki
clf_model = None  # TODO

y_pred_clf = None  # TODO

acc = None
prec = None
rec = None
f1score = None

print("Accuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1score)

# TODO: w komentarzu napisz, która z metryk jest Twoim zdaniem najważniejsza dla tego case (i dlaczego).


## Zadanie 13 – Bias vs variance na wykresie (dla chętnych)

Opis biznesowy:
- Na prostym przykładzie (np. regresja 1D) zobaczysz, jak zmienia się błąd na zbiorze treningowym i testowym, gdy zwiększa się złożoność modelu.

Cel techniczny:
- Zobaczyć wizualnie zjawiska underfitting i overfitting.

Poziom: 3 (trudniejsze)

Szacowany czas: 45–60 minut

Instrukcja (propozycja):
- Wygeneruj dane 1D z lekkim szumem (np. x w [0, 1], y = sin(2 pi x) + noise).
- Dla rosnącego stopnia wielomianu (np. degree = 1, 3, 5, 9):
  - dopasuj model regresji wielomianowej,
  - policz MSE na train i test,
  - zapisz wyniki.
- Narysuj wykres: stopień wielomianu vs błąd (osobne linie dla train i test).
- Zinterpretuj, gdzie jest underfitting, a gdzie overfitting.

W tym zadaniu możesz użyć bardziej zaawansowanych narzędzi `scikit-learn` (np. `PolynomialFeatures`, `Pipeline`), ale nie jest to konieczne – możesz także korzystać z prostszych rozwiązań.


In [ ]:
# Zadanie 13 – Bias vs variance na wykresie (dla chętnych)

# TODO: jeśli chcesz zrealizować to zadanie, wykonaj następujące kroki:
# 1. Wygeneruj dane 1D (X, y) z jakąś prostą funkcją + noise.
# 2. Dla kilku poziomów złożoności modelu policz błąd na train i test.
# 3. Narysuj wykres błędu vs złożoność.
# 4. W komentarzu opisz, które modele mają wysoki bias, a które wysoką variance.

# Podpowiedź: możesz użyć PolynomialFeatures + LinearRegression + Pipeline.
# from sklearn.preprocessing import PolynomialFeatures
# from sklearn.linear_model import LinearRegression
# from sklearn.pipeline import Pipeline
# from sklearn.metrics import mean_squared_error
# from sklearn.model_selection import train_test_split

# TODO: zaimplementuj swoją wersję eksperymentu bias-variance.

import matplotlib.pyplot as plt

# Tutaj możesz dodać swój kod.

pass  # TODO: usuń pass i wstaw własny kod eksperymentu
